# Same as linear_testing_2.ipynb, but fitting the pseudo-scores with each TS

In [1]:
from astropy.io import fits

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy import stats as ss
from scipy.io import readsav
from sklearn.decomposition import PCA

from tqdm import tqdm

import tools as t

## Load data

In [2]:
# load files
path_data = '../data/sim_series/'  # directory where the files are saved
N_files = 20  # 20 temporal series (from 0 to 19)
data_spectra = []
data_rv = []
for i in range(N_files):  # files have the same name, only number changes
    data_spectra.append(fits.getdata(path_data + f'flux_norm_6173_lionel_tau1_spatialscaling1_sansrotation_serie{i}.fits'))
    data_rv.append(readsav(path_data + f'res_rv_bis_6173_lionel_tau1_scaling1_sansrot_serie{i}.sav'))
data_spectra = np.array(data_spectra)
data_rv = np.array(data_rv)
print(data_rv[0].keys())

dict_keys(['rv_gauss', 'rv_correl', 'rv_deriv', 'xbiss', 'ybiss', 'l_true', 'lambd', 'flux_moy'])


In [3]:
c = 3e8  # speed of light [m/s]
# extract data
wl = np.array(data_rv[0]['lambd'])  # same wavelength grid for all time series (verified)
times = np.arange(0, len(data_spectra[0]) * 144, 144) / 60 / 24  # time in days, same for all time series
times_sec = times * 86400  # time in sec
# extract RV data
wl = np.array(wl)

## Work with only one TS

In [4]:
# selecting TS
ts_index = 4

spectra_ex = data_spectra[ts_index]

# splitting TS in two halves
N = len(times)
N_half = N // 2

times_half = times[:N_half]

spectra_0 = spectra_ex[:N_half]
spectra_1 = spectra_ex[N_half:]

## Preprocessing

In [5]:
# calculate the mean spectra
mean_spec_0 = np.mean(spectra_0, axis=0)
mean_spec_1 = np.mean(spectra_1, axis=0)

d_S0 = np.gradient(mean_spec_0, wl)
d_S1 = np.gradient(mean_spec_1, wl)

In [6]:
Sf_0, D_0 = t.subtract_D(wl, spectra_0, c=c)  # S_f and Doppler vector
Sf_1, D_1 = t.subtract_D(wl, spectra_1, c=c)

spectra_centered_0 = t.double_centering(Sf_0)
spectra_centered_1 = t.double_centering(Sf_1)

In [7]:
# preprocessing all TS
mean_spec = np.array([np.mean(i, axis=0) for i in data_spectra])  # list of mean spectrum for each TS
d_S0 = np.array([np.gradient(i, wl) for i in mean_spec])  # list of derivative of the mean spectra for each TS

S_f = []  # array of matrices
D_vectors = []  # array of Doppler vectors

for i in range(N_files):  # for each TS
    s, v = t.subtract_D(wl, data_spectra[i], c=c)  # take out projection over the Doppler vector
    S_f.append(s)
    D_vectors.append(v)

S_f = np.array(S_f)
D_vectors = np.array(D_vectors)

spectra_centered = np.array([t.double_centering(i) for i in S_f])

## Calculate RV by template matching

In [8]:
# calculate radial velocity by template matching
C_0 = 1000  # continuum constant

# list of RVs for each timestep
rv_0 = np.array([
    t.calculate_rv(wl, mean_spec_0, spectra_0[i], C_0) for i in range(N_half)
    ])

rv_1 = np.array([
    t.calculate_rv(wl, mean_spec_1, spectra_1[i], C_0) for i in range(N_half)
    ])

In [9]:
# calculate RV for all the TS

rv_tm = []  # list of RV vector for each TS
for j in range(N_files):  # for each TS
    rv_j = []
    for k in range(len(times)):  # for each spectrum
        rv_j.append(t.calculate_rv(wl, mean_spec[j], data_spectra[j][k], C_0))  # calculate RV by TM
    rv_tm.append(np.array(rv_j))
rv_tm = np.array(rv_tm)

## PCA

In [10]:
# apply PCA to the centered matrix
n_components = 16

# first half
pca_flux_0 = PCA(n_components=n_components)           # create PCA object
pcs_0 = pca_flux_0.fit_transform(spectra_centered_0)  # fit the model with spectra_centered and apply the dimensionality reduction
loadings_0 = pca_flux_0.components_                   # save loadings
scores_0 = pcs_0.T                                    # save scores
var_0 = pca_flux_0.explained_variance_ratio_          # save PC variation

# second half
pca_flux_1 = PCA(n_components=n_components)           # create PCA object
pcs_1 = pca_flux_1.fit_transform(spectra_centered_1)  # fit the model with spectra_centered and apply the dimensionality reduction
loadings_1 = pca_flux_1.components_                   # save loadings
scores_1 = pcs_1.T                                    # save scores
var_1 = pca_flux_1.explained_variance_ratio_          # save PC variation

In [11]:
print(f'Centred TS of spectra shape: {spectra_centered_0.shape}')
print(f'Loadings shape: {loadings_0.shape}')
print(f'Scores shape: {scores_0.shape}')

Centred TS of spectra shape: (1843, 122)
Loadings shape: (16, 122)
Scores shape: (16, 1843)


In [12]:
# pseudo-scores for training
proj_0 = []
for i in range(n_components):
    proj_i = []
    loadings_i = loadings_0[i]
    for j in range(N_half):
        proj_ij = np.dot(loadings_i.T, spectra_centered_0[j]) / np.linalg.norm(loadings_i)**2
        proj_i.append(proj_ij)
    proj_0.append(proj_i)
proj_0 = np.array(proj_0)

## Linear Model

### TS1A

In [ ]:
# train/test split
train_size = int(.7 * N_half)
I_train_0 = proj_0.T[:train_size]  # using pseudo-scores
rv_true_train_0 = rv_0[:train_size]
times_train = times_half[:train_size]

I_test_0 = proj_0.T[train_size:]  # using pseudo-scores
spectra_centered_test_0 = spectra_centered_0[train_size:]
rv_true_test_0 = rv_0[train_size:]
times_test = times_half[train_size:]

# training the model
G = np.dot(I_train_0.T, I_train_0)
b = np.dot(I_train_0.T, rv_true_train_0)
alpha = np.linalg.solve(G, b)

In [14]:
# testing on the test sample of the same half

chunk_size = 21   # using chunks to get error bars

rv_pred_test_0 = []    # list of chunks of predicted RV
res_test_0 = []        # list of chunks of residuals calculated from rv_pred
res_sigma_test_0 = []  # list of numbers: std of each chunk in res

j = 0
while j < len(rv_true_test_0):  # iterate over chunks
    if j + chunk_size > len(rv_true_test_0):  # last elements of the TS not included if (N % chunk_size != 0)
        break

    else:
        pseudo_scores_j = I_test_0[j:j+chunk_size]  # pseudo-scores of shape chunk_size x PCs
        rv_true_j = rv_true_test_0[j:j+chunk_size]  # chunk_size values of RV
        
        rv_pred_j = np.dot(pseudo_scores_j, alpha)  # predict chunk_size values of RV
        res_j = rv_true_j - rv_pred_j  # chunk residuals

        rv_pred_test_0.append(rv_pred_j)        # append chunk of resulting RV to a list
        res_test_0.append(res_j)                # append residuals of each chunk
        res_sigma_test_0.append(np.std(res_j))  # append std of res_j
    j += chunk_size

rv_pred_test_0 = np.array(rv_pred_test_0)
res_test_0 = np.array(res_test_0)
res_test_0_flat = res_test_0.flatten()
res_sigma_test_0 = np.array(res_sigma_test_0)

In [15]:
print(f'The standard deviation of the residuals is {np.std(res_test_0):.3f} m/s.')  # using pseudo-scores gives the ssame result as using scores

The standard deviation of the residuals is 0.189 m/s.


### TS1B

In [16]:
# calculating pseudo-scores for the second half
proj_1 = []
for i in range(n_components):
    proj_i = []
    loadings_i = loadings_0[i]  # using loadings from the first half
    for j in range(N_half):
        proj_ij = np.dot(loadings_i.T, spectra_centered_1[j]) / np.linalg.norm(loadings_i)**2  # projecting spectra from the second half on the loadings from the first half
        proj_i.append(proj_ij)
    proj_1.append(proj_i)
proj_1 = np.array(proj_1)

# train/test split
I_train_1 = proj_1.T[:train_size]  # using pseudo-scores
rv_true_train_1 = rv_1[:train_size]
times_train = times_half[:train_size]

I_test_1 = proj_1.T[train_size:]  # using pseudo-scores
spectra_centered_test_1 = spectra_centered_1[train_size:]
rv_true_test_1 = rv_1[train_size:]
times_test = times_half[train_size:]

# training the model
G = np.dot(I_train_1.T, I_train_1)
b = np.dot(I_train_1.T, rv_true_train_1)
alpha_1 = np.linalg.solve(G, b)  # geting alpha by using the pseudo-scores

In [17]:
# testing on the whole second half
rv_pred_1 = []    # list of chunks of predicted RV
res_1 = []        # list of chunks of residuals calculated from rv_pred
res_sigma_1 = []  # list of numbers: std of each chunk in res

j = 0
while j < N_half:  # iterate over chunks
    if j + chunk_size > N_half:  # last elements of the TS not included if (N % chunk_size != 0)
        break

    else:
        pseudo_scores_j = proj_1.T[j:j+chunk_size]  # pseudo-scores of shape chunk_size x PCs
        rv_true_j = rv_1[j:j+chunk_size]  # chunk_size values of RV

        rv_pred_j = np.dot(pseudo_scores_j, alpha_1)  # predict chunk_size values of RV
        res_j = rv_true_j - rv_pred_j  # chunk residuals

        rv_pred_1.append(rv_pred_j)        # append chunk of resulting RV to a list
        res_1.append(res_j)                # append residuals of each chunk
        res_sigma_1.append(np.std(res_j))  # append std of res_j
    j += chunk_size

rv_pred_1 = np.array(rv_pred_1)
res_1 = np.array(res_1)
res_1_flat = res_1.flatten()
res_sigma_1 = np.array(res_sigma_1)

In [18]:
print(f'The standard deviation of the residuals is {np.std(res_1):.3f} m/s.')

The standard deviation of the residuals is 0.186 m/s.


## Testing on all other TS

In [19]:
# create pseudo-scores fo each TS, using spectra_centered and loadings_0
pseudo_scores = []

for k in range(N_files):  # iterate over TS
    spectra_centered_k = spectra_centered[k]
    proj_k = []
    for i in range(n_components):  # iterate over PCs
        proj_i = []
        loadings_i = loadings_0[i]  # using reference loadings
        for j in range(N):  # iterate over timesteps
            proj_ij = np.dot(loadings_i.T, spectra_centered_k[j]) / np.linalg.norm(loadings_i)**2  # projecting spectra on the reference loadings
            proj_i.append(proj_ij)
        proj_k.append(proj_i)
    pseudo_scores.append(proj_k)

pseudo_scores = np.array(pseudo_scores)

In [ ]:
train_size =   # check this for compatibility with the half-series train-test split

# train-test split for each TS
I_train = []
rv_true_train = []
#times_train = times[:train_size]  # check this for compatibility with the half-series train-test split

I_test = []
spectra_centered_test = []
rv_true_test = []
#times_test = times[test_size:]

for i in range(N_files):
    pseudo_scores_i = pseudo_scores[i]
    I_train.append(pseudo_scores[:train_size])  # using pseudo-scores
    rv_true_train_1 = rv_1[:train_size]
    times_train = times_half[:train_size]

    I_test_1 = proj_1.T[train_size:]  # using pseudo-scores
    spectra_centered_test_1 = spectra_centered_1[train_size:]
    rv_true_test_1 = rv_1[train_size:]
    times_test = times_half[train_size:]

In [ ]:
# training for each TS
alpha_ts = []

for i in range(N_files):
    pseudo_scores_i = pseudo_scores[i]
    rv_true_i = rv_tm[i]

In [ ]:
# testing with chunks
#max_len = (N // chunk_size) * chunk_size

rv_pred = []
res = []
res_sigma = []

for i in range(N_files):  # iterate over TS

    pseudo_scores_i = pseudo_scores[i]
    rv_true_i = rv_tm[i]

    rv_pred_i = []    # list of chunks of predicted RV
    res_i = []        # list of chunks of residuals calculated from rv_pred_i
    res_sigma_i = []  # list of numbers: std of each chunk in res_i

    j = 0

    while j < N:  # iterate over chunks

        if j + chunk_size > N:  # last elements of the TS not included if (N % chunk_size != 0)
            break

        else:
            pseudo_scores_j = pseudo_scores_i.T[j:j+chunk_size]  # pseudo-scores of shape chunk_size x PCs
            rv_true_j = rv_true_i[j:j+chunk_size]  # chunk_size values of RV

            rv_pred_j = np.dot(pseudo_scores_j, alpha)  # predict chunk_size values of RV
            res_j = rv_true_j - rv_pred_j  # chunk residuals
            
            rv_pred_i.append(rv_pred_j)        # append chunk of resulting RV to a list
            res_i.append(res_j)                # append residuals of each chunk
            res_sigma_i.append(np.std(res_j))  # append std of res_j
        
        j += chunk_size

    rv_pred.append(np.array(rv_pred_i))
    res.append(np.array(res_i))
    res_sigma.append(np.array(res_sigma_i))

rv_pred = np.array(rv_pred)
res = np.array(res)
res_sigma = np.array(res_sigma)